In [8]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta

# ==========================================
# 1. DOWNLOAD DATA
# ==========================================
ticker = "CNH=X"  # Yahoo Finance ticker for USD/CNH
interval = "15m"
period = "60d"    # Yahoo Finance limits 15m data to the last 360 days

print(f"Downloading {interval} data for {ticker}...")
df = yf.download(ticker, period=period, interval=interval)

# Flatten MultiIndex columns if using recent versions of yfinance
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.droplevel(1)

# Keep only the necessary columns
df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

# ==========================================
# 2. FEATURE ENGINEERING (Using 'ta' library)
# ==========================================
print("Calculating technical indicators...")

# 1. Price Action: Log Returns
df['Log_Returns'] = np.log(df['Close'] / df['Close'].shift(1))

# 2. Momentum: 14-period RSI
df['RSI_14'] = ta.momentum.RSIIndicator(close=df['Close'], window=14).rsi()

# 3. Momentum: MACD (Fast 12, Slow 26, Signal 9)
macd_indicator = ta.trend.MACD(close=df['Close'], window_slow=26, window_fast=12, window_sign=9)
df['MACD_Line'] = macd_indicator.macd()
# Optional: If you also want the MACD histogram or signal line as inputs:
# df['MACD_Signal'] = macd_indicator.macd_signal()
# df['MACD_Diff'] = macd_indicator.macd_diff()

# 4. Volatility: 14-period ATR
df['ATR_14'] = ta.volatility.AverageTrueRange(high=df['High'], low=df['Low'], close=df['Close'], window=14).average_true_range()

# 5. Time/Session: Sine of the Hour
df['Hour_Sin'] = np.sin(2 * np.pi * df.index.hour / 24)

# ==========================================
# 3. CREATE TARGET VARIABLE
# ==========================================
# Target = 1 if the next 15m candle closes higher, else 0
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

# ==========================================
# 4. CLEAN AND SAVE
# ==========================================
# Drop NaN values generated by the lookback periods (like the 26-period MACD)
df.dropna(inplace=True)

# Save to CSV
filename = "USD_CNH_15m_ML_ready.csv"
df.to_csv(filename)

print(f"Success! {len(df)} rows of data saved to {filename}")

[*********************100%***********************]  1 of 1 completed

Calculating technical indicators...
Success! 5623 rows of data saved to USD_CNH_15m_ML_ready.csv


In [ ]:
import pandas as pd
import numpy as np
import ta

# ==========================================
# 1. LOAD MT5 DATA
# ==========================================
# Replace with your actual MT5 exported file name
input_file = "USDCNH_M15_202401020000_202604102345.csv"

print(f"Loading data from {input_file}...")
# MT5 exports are tab-separated, so we must specify sep='\t'
df = pd.read_csv(input_file, sep='\t')

# Combine Date and Time into a single Datetime index
df['Datetime'] = pd.to_datetime(df['<DATE>'] + ' ' + df['<TIME>'])
df.set_index('Datetime', inplace=True)

# Rename columns to standard readable names
df = df.rename(columns={
    '<OPEN>': 'Open',
    '<HIGH>': 'High',
    '<LOW>': 'Low',
    '<CLOSE>': 'Close',
    '<TICKVOL>': 'Volume'
})

# Keep only the necessary columns
df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

# ==========================================
# 2. FEATURE ENGINEERING (Using 'ta' library)
# ==========================================
print("Calculating technical indicators...")

# 1. Price Action: Log Returns
df['Log_Returns'] = np.log(df['Close'] / df['Close'].shift(1))

# 2. Momentum: 14-period RSI
df['RSI_14'] = ta.momentum.RSIIndicator(close=df['Close'], window=14).rsi()

# 3. Momentum: MACD Line (Fast 12, Slow 26, Signal 9)
macd_indicator = ta.trend.MACD(close=df['Close'], window_slow=26, window_fast=12, window_sign=9)
df['MACD_Line'] = macd_indicator.macd()

# 4. Volatility: 14-period ATR
df['ATR_14'] = ta.volatility.AverageTrueRange(high=df['High'], low=df['Low'], close=df['Close'], window=14).average_true_range()

# 5. Time/Session: Sine of the Hour
df['Hour_Sin'] = np.sin(2 * np.pi * df.index.hour / 24)

# ==========================================
# 3. CREATE TARGET VARIABLE
# ==========================================
# Target = 1 if the next 15m candle closes higher, else 0
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

# ==========================================
# 4. CLEAN AND SAVE
# ==========================================
# Drop NaN values generated by the lookback periods (e.g., first 26 periods for MACD)
df.dropna(inplace=True)

# Save to CSV
output_file = "USDCNH_15m_ML_ready_2years.csv"
df.to_csv(output_file)

print(f"Success! {len(df)} rows of data saved to {output_file}")